# 2D Swift-Hohenberg Equation
## Pattern Formation, Stripes, and Hexagons

This notebook simulates the 2D Swift-Hohenberg equation using the pseudo-spectral `PDESolver` framework. Originally derived to model Rayleigh-Bénard convection (fluid heated from below), it is the quintessential equation for studying spontaneous pattern formation, Turing instabilities, and localized structures (like "spots" or "bugs") in non-equilibrium systems.

---

## 1. The Governing Equation

$$
\partial_t u = r u - u^3 - (\nabla^2 + 1)^2 u
$$

* $r$ (Bifurcation Parameter): Controls the distance from the instability threshold. When $r > 0$, the uniform state $u=0$ becomes unstable to perturbations with a specific preferred wavenumber.
* $-u^3$ (Nonlinear Saturation): Limits the growth of the pattern, preventing blow-up and selecting the final amplitude.
* $-(\nabla^2 + 1)^2$ (Linear Operator): This is the magic term. It acts as a band-pass filter in Fourier space. It strongly damps both very large scales (small $k$) and very small scales (large $k$), but *amplifies* modes near the critical wavenumber $k_c = 1$ (wavelength $\lambda = 2\pi$).

---

## 2. Reformulation for the Solver

We expand the linear operator to identify its Fourier symbol. In Fourier space, $\nabla^2 \to -k^2 = -(\xi^2 + \eta^2)$.

$$
-(\nabla^2 + 1)^2 \longrightarrow -(-k^2 + 1)^2 = -(1 - k^2)^2
$$

The full linear symbol is therefore:

$$
\text{Linear symbol:} \quad r - (1 - (\xi^2 + \eta^2))^2
$$

The equation in the solver's format:

$$
\partial_t u = \underbrace{u_{\text{op}}\!\left(r - (1 - (\xi^2 + \eta^2))^2\right)u} {\text{Linear band-pass filter (Fourier space)}} \underbrace{- u^3} {\text{Nonlinear saturation (Physical space)}}
$$

---

## 3. Physical Phenomena

* **Spontaneous Symmetry Breaking**: Starting from a uniform state with tiny random noise, the system spontaneously organizes into regular patterns.
* **Stripes and Hexagons**: Depending on the exact parameters and initial conditions, the system settles into labyrinthine stripes or hexagonal arrays of spots.
* **Defect Dynamics**: As domains of different orientations collide, topological defects (dislocations and disclinations) form and slowly annihilate over long timescales.

We initialize the system with a **uniform state plus small random noise** to trigger the instability and watch the beautiful self-organization unfold.

# Implementation
## 0. Imports 

In [ ]:
from solver import PDESolver, psiOp  # psiOp for real-valued fields
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters 

In [ ]:
# ── Swift-Hohenberg Coefficients ──
R = 1.5       # Bifurcation parameter (r > 0 triggers pattern formation)

# ── Grid and Time ──
# The preferred wavelength is lambda = 2*pi ≈ 6.28.
# We need a large domain to fit multiple wavelengths and observe domain coarsening.
Lx, Ly = 60.0, 60.0

# Nx, Ny = 128, 128    
# Nx, Ny = 256, 256    # Higher resolution for sharp interfaces
Nx, Ny = 512, 512    # Higher resolution for sharp interfaces

# Pattern formation is slow; we need a long integration time
Lt, Nt = 50.0, 1000
n_frames = 200

## 2. Grid setup 

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol 

In [ ]:
t, x, y   = sp.symbols('t x y', real=True)
xi, eta   = sp.symbols('xi eta', real=True)
u_func    = sp.Function('u')
u_field   = u_func(t, x, y)

# ── Linear symbol in Fourier space ──
# From: ∂u/∂t = r·u - (∇² + 1)²u + ...
# Fourier: ∇² → -(ξ² + η²)
# So: r - (-(ξ² + η²) + 1)² = r - (1 - (ξ² + η²))²

k2 = xi**2 + eta**2
symbol_linear = R - (1 - k2)**2

print('Principal symbol (linear part):')
print('  a(ξ, η) = ', symbol_linear)

## 4. Swift-Hohenberg equation 

In [ ]:
# ∂u/∂t = psiOp(r - (1 - k²)², u)  -  u³
#        ───────────────────────    ────
#        Linear band-pass filter    Nonlinear saturation

equation = sp.Eq(
    sp.diff(u_field, t),
    psiOp(symbol_linear, u_field) - u_field**3
)

print('Swift-Hohenberg Equation:')
print('  ∂u/∂t = psiOp(r - (1 - k²)², u) - u³')

## 5. Initial conditions: Random noise 

In [ ]:
def initial_condition_sh(xx, yy):
    """
    Uniform state (u=0) with small random perturbations.
    The linear instability will amplify the noise at the preferred wavenumber k=1.
    """
    np.random.seed(420)  # For reproducibility
    noise_amplitude = 0.1
    return noise_amplitude * np.random.randn(xx.shape[0], xx.shape[1])

## 6. Solver setup 

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',  # Periodic BCs are standard for pattern formation
    initial_condition=initial_condition_sh,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve 

In [ ]:
frames = solver.solve()

## 8. Visualization 

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',    # Show Re(u) (the physical field)
    overlay=None,  
    mode='surface',      # 'surface' or 'contour' both look great for SH
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
# ani.save('swift_hohenberg_pattern_formation.mp4', writer='ffmpeg', fps=20, dpi=100)
# print('✅ Saved to swift_hohenberg_pattern_formation.mp4')